# Depuración de Cargos — Construcción de tabla operativa (Kactus + Consolidado 2025)

Notebook único y reproducible. **Fase 1** (secciones 1–9) consolida los 8
archivos Kactus de `Insumos_Vigentes` como DataFrame en memoria — no exporta
ningún Excel intermedio. **Fase 2** (secciones 10 en adelante) enriquece esa
tabla con `Consolidado 2025.xlsx` (julio 2026) y exporta el **único** archivo
de salida del pipeline: `Outputs/Depuracion_Cargos/Tabla_Depuracion_Cargos.xlsx`,
con 3 hojas (`Depuracion_Cargos`, `QA_Conciliacion`,
`Conflictos_Dependencia_Area`).

## Objetivo

Construir una tabla operativa de **cargos** (no de empleados) a partir de los
8 archivos Excel de Kactus en `Data/Maestro_Cargos-Roles_Kactus/Insumos_Vigentes/`,
de forma reproducible y auditable, sin modificar los archivos fuente.

## Reglas de negocio de esta fase

- Granularidad candidata: `Empresa + Cargo` (se valida más abajo, no se asume).
- No se eliminan duplicados de forma automática. Si existen, se documentan y
  se dejan visibles en la tabla final para decisión posterior.
- No se filtra por `Ind. Actividad` (activos/inactivos) — esta tabla es
  precisamente para depuración.
- `Dependencia`, `Área` y `Fecha de actualización` se dejan vacías en la
  Fase 1; se completan en la Fase 2 con `Consolidado 2025.xlsx` (julio 2026)
  y con la fecha real de cierre de la tarea completa.
- Este notebook NO escribe en `Data/` ni en ningún archivo trackeado por Git.

## Privacidad

Las celdas de este notebook solo deben imprimir **métricas agregadas** de QA
(conteos, nombres de archivo, nombres de columnas). No se debe imprimir ni
dejar como output ninguna fila individual con datos de cargos.


## 1. Configuración y rutas

In [ ]:
from pathlib import Path
import sys
import datetime as dt

import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font
from openpyxl.worksheet.table import Table, TableStyleInfo


def find_repo_root(start: Path) -> Path:
    """Sube directorios hasta encontrar la raíz del repositorio (marcador: AGENTS.md + PBIP/)."""
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "PBIP").is_dir():
            return candidate
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio (se esperaba AGENTS.md y PBIP/ en algún ancestro)."
    )


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "Data" / "Maestro_Cargos-Roles_Kactus" / "Insumos_Vigentes"
OUTPUT_DIR = REPO_ROOT / "Outputs" / "Depuracion_Cargos"
# Único archivo Excel que produce este notebook. La Fase 1 se mantiene como DataFrame
# en memoria (sin exportar un Excel intermedio); todo se exporta aquí al cierre de la Fase 2.
TABLA_FINAL_XLSX = OUTPUT_DIR / "Tabla_Depuracion_Cargos.xlsx"

EXPECTED_SOURCE_FILES = 8

# Fase actual: NO se enriquece con Consolidado 2025.xlsx todavía.
# Dependencia / Área / Fecha de actualización quedan vacías en esta entrega.
FECHA_ACTUALIZACION = None  # se define solo en la ejecución final completa de toda la tarea

REQUIRED_SOURCE_COLUMNS = [
    "Empresa",
    "Nombre Empresa",
    "Cargo",
    "Fecha de Creación",
    "Ind. Actividad",
    "Nombre del Cargo",
    "Número de Cargos",
    "Cargos Ocupados",
]

FINAL_COLUMNS = [
    "Número de cargo",
    "Fecha de creación",
    "Ind. Actividad",
    "Nombre del cargo",
    "Número de cargos",
    "Cargos ocupados",
    "Dependencia",
    "Área",
    "Fecha de actualización",
    "Empresa",
    "Nombre Empresa",
]

print(f"REPO_ROOT        = {REPO_ROOT}")
print(f"SRC_DIR          = {SRC_DIR}")
print(f"OUTPUT_DIR       = {OUTPUT_DIR}")
print(f"TABLA_FINAL_XLSX = {TABLA_FINAL_XLSX}")
print(f"SRC_DIR existe: {SRC_DIR.is_dir()}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Inventario de fuentes

In [ ]:
import openpyxl

all_entries = sorted(p.name for p in SRC_DIR.iterdir())
xlsx_files = sorted(SRC_DIR.glob("*.xlsx"))

print(f"Entradas totales en la carpeta: {len(all_entries)}")
for name in all_entries:
    print(f"  - {name}")

print()
print(f"Archivos .xlsx encontrados: {len(xlsx_files)}")
print(f"Archivos .xlsx esperados:   {EXPECTED_SOURCE_FILES}")

if len(xlsx_files) != EXPECTED_SOURCE_FILES:
    print(
        f"ALERTA: se esperaban {EXPECTED_SOURCE_FILES} archivos y se encontraron "
        f"{len(xlsx_files)}. No se asume nada: revisar manualmente antes de continuar."
    )
else:
    print("OK: coincide con los 8 archivos esperados.")

print()
print("Nombres de archivo procesados en esta ejecución:")
for f in xlsx_files:
    print(f"  - {f.name}")


## 3. Perfilado de los 8 Excel

Se valida, por archivo: número de hojas, dimensiones, y presencia exacta de
las columnas fuente requeridas (por nombre, no por posición).


In [ ]:
profile_rows = []

for f in xlsx_files:
    wb = openpyxl.load_workbook(f, read_only=True, data_only=True)
    sheet_names = wb.sheetnames
    ws = wb[sheet_names[0]]
    header = list(next(ws.iter_rows(min_row=1, max_row=1, values_only=True)))
    n_rows = ws.max_row - 1 if ws.max_row and ws.max_row >= 1 else 0
    n_cols = ws.max_column
    missing_cols = [c for c in REQUIRED_SOURCE_COLUMNS if c not in header]
    wb.close()

    profile_rows.append({
        "archivo": f.name,
        "n_hojas": len(sheet_names),
        "hoja_usada": sheet_names[0],
        "filas_datos": n_rows,
        "columnas": n_cols,
        "columnas_requeridas_faltantes": missing_cols,
    })

    if len(sheet_names) > 1:
        print(f"ALERTA: {f.name} tiene {len(sheet_names)} hojas: {sheet_names}. Se usa la primera ('{sheet_names[0]}') — confirmar si es correcto.")
    if missing_cols:
        print(f"ALERTA: {f.name} no tiene estas columnas requeridas: {missing_cols}")

profile_df = pd.DataFrame(profile_rows)
print()
print(profile_df.to_string(index=False))

total_filas_por_archivo = int(profile_df["filas_datos"].sum())
print()
print(f"Total de filas de datos sumando los {len(xlsx_files)} archivos (antes de deduplicar): {total_filas_por_archivo}")


## 4. Carga y estandarización

Se cargan las columnas requeridas de cada archivo, se renombran al esquema
final y se normalizan tipos de datos:

- `Número de cargo`, `Empresa` → texto (evitar pérdida de ceros a la izquierda).
- `Fecha de creación` → fecha.
- `Número de cargos`, `Cargos ocupados` → numérico.
- `Ind. Actividad`, `Nombre del cargo`, `Nombre Empresa` → texto, tal como viene de Kactus.

Se conserva temporalmente una columna interna `__archivo_origen__` solo para
trazabilidad de QA dentro de este notebook; se elimina antes de exportar.


In [ ]:
COLUMN_RENAME = {
    "Cargo": "Número de cargo",
    "Fecha de Creación": "Fecha de creación",
    "Ind. Actividad": "Ind. Actividad",
    "Nombre del Cargo": "Nombre del cargo",
    "Número de Cargos": "Número de cargos",
    "Cargos Ocupados": "Cargos ocupados",
    "Empresa": "Empresa",
    "Nombre Empresa": "Nombre Empresa",
}

frames = []
filas_por_archivo = {}

for f in xlsx_files:
    df_raw = pd.read_excel(f, sheet_name=0, dtype=object, engine="openpyxl")
    filas_por_archivo[f.name] = len(df_raw)

    faltantes = [c for c in REQUIRED_SOURCE_COLUMNS if c not in df_raw.columns]
    if faltantes:
        raise ValueError(f"{f.name}: faltan columnas requeridas {faltantes}, no se puede continuar de forma segura.")

    df_sel = df_raw[REQUIRED_SOURCE_COLUMNS].rename(columns=COLUMN_RENAME).copy()
    df_sel["__archivo_origen__"] = f.name
    frames.append(df_sel)

df_all = pd.concat(frames, ignore_index=True)

print("Filas leídas por archivo:")
for nombre, n in filas_por_archivo.items():
    print(f"  - {nombre}: {n}")

print()
print(f"Filas consolidadas (antes de deduplicar): {len(df_all)}")
assert len(df_all) == sum(filas_por_archivo.values()), "La suma de filas por archivo no coincide con el total consolidado."


In [ ]:
def to_text_id(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value).strip()


def to_number(value):
    if value is None or (isinstance(value, str) and value.strip() == ""):
        return None
    try:
        f_val = float(value)
    except (TypeError, ValueError):
        return None
    if f_val.is_integer():
        return int(f_val)
    return f_val


def is_blank(value):
    if value is None:
        return True
    if isinstance(value, float) and pd.isna(value):
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def to_date(value):
    if is_blank(value):
        return None
    if isinstance(value, (dt.datetime, dt.date)):
        return pd.Timestamp(value).normalize()
    parsed = pd.to_datetime(value, errors="coerce", dayfirst=True)
    if pd.isna(parsed):
        return None
    return parsed.normalize()


df_std = df_all.copy()

# Detectar códigos de cargo/empresa que llegan como número (riesgo de ceros a la izquierda perdidos en la fuente)
numeric_cargo_mask = df_std["Número de cargo"].apply(lambda v: isinstance(v, (int, float)) and not isinstance(v, bool))
numeric_empresa_mask = df_std["Empresa"].apply(lambda v: isinstance(v, (int, float)) and not isinstance(v, bool))

df_std["Número de cargo"] = df_std["Número de cargo"].apply(to_text_id)
df_std["Empresa"] = df_std["Empresa"].apply(to_text_id)
df_std["Nombre del cargo"] = df_std["Nombre del cargo"].apply(lambda v: str(v).strip() if v is not None else None)
df_std["Nombre Empresa"] = df_std["Nombre Empresa"].apply(lambda v: str(v).strip() if v is not None else None)
df_std["Ind. Actividad"] = df_std["Ind. Actividad"].apply(lambda v: str(v).strip() if v is not None else None)

def fecha_es_valida_o_vacia(v):
    """Vacia = sin dato (no es un error de formato). Invalida = tiene un valor que no es fecha."""
    if is_blank(v):
        return True
    if isinstance(v, (dt.datetime, dt.date)):
        return True
    if isinstance(v, str):
        return not pd.isna(pd.to_datetime(v, errors="coerce", dayfirst=True))
    return False


fecha_vacia_mask = df_std["Fecha de creación"].apply(is_blank)
fecha_valida_mask = df_std["Fecha de creación"].apply(fecha_es_valida_o_vacia)
df_std["Fecha de creación"] = df_std["Fecha de creación"].apply(to_date)

n_cargos_valido_mask = df_std["Número de cargos"].apply(lambda v: is_blank(v) or isinstance(v, (int, float)))
cargos_ocup_valido_mask = df_std["Cargos ocupados"].apply(lambda v: is_blank(v) or isinstance(v, (int, float)))
df_std["Número de cargos"] = df_std["Número de cargos"].apply(to_number)
df_std["Cargos ocupados"] = df_std["Cargos ocupados"].apply(to_number)

print(f"Códigos de 'Número de cargo' que llegaron como valor numérico en la fuente (riesgo de ceros a la izquierda): {int(numeric_cargo_mask.sum())}")
print(f"Códigos de 'Empresa' que llegaron como valor numérico en la fuente: {int(numeric_empresa_mask.sum())}")


## 5. QA de granularidad

Se valida la clave de unión candidata (`Empresa + Cargo`) antes de asumirla:
unicidad de `Cargo` solo, unicidad de `Empresa + Cargo`, duplicados exactos y
duplicados con atributos distintos para la misma clave.


In [ ]:
n_total = len(df_std)
n_cargo_solo_unico = df_std["Número de cargo"].nunique(dropna=True)
n_empresa_cargo_unico = df_std.dropna(subset=["Empresa", "Número de cargo"]).drop_duplicates(subset=["Empresa", "Número de cargo"]).shape[0]

print(f"Filas totales: {n_total}")
print(f"Valores únicos de 'Número de cargo' (ignorando Empresa): {n_cargo_solo_unico}")
print(f"Combinaciones únicas de 'Empresa + Número de cargo': {n_empresa_cargo_unico}")

if n_cargo_solo_unico < n_total:
    print("CONFIRMADO: 'Número de cargo' NO es único globalmente (se repite entre empresas) — no debe usarse solo como clave.")

target_cols = [
    "Número de cargo", "Fecha de creación", "Ind. Actividad", "Nombre del cargo",
    "Número de cargos", "Cargos ocupados", "Empresa", "Nombre Empresa",
]

# Duplicados exactos: mismas 8 columnas de negocio en más de una fila
dup_exact_mask = df_std.duplicated(subset=target_cols, keep=False)
n_dup_exact_rows = int(dup_exact_mask.sum())
n_dup_exact_groups = df_std[dup_exact_mask].drop_duplicates(subset=target_cols).shape[0]
print()
print(f"Filas involucradas en duplicados EXACTOS (mismas 8 columnas de negocio): {n_dup_exact_rows}")
print(f"Grupos distintos de duplicado exacto: {n_dup_exact_groups}")

# Duplicados por Empresa+Cargo con posibles diferencias en otros atributos
key_cols = ["Empresa", "Número de cargo"]
grp_sizes = df_std.dropna(subset=key_cols).groupby(key_cols).size()
claves_repetidas = grp_sizes[grp_sizes > 1]
print()
print(f"Claves 'Empresa + Número de cargo' con más de una fila: {len(claves_repetidas)}")

n_claves_repetidas_no_exactas = 0
for (empresa, cargo), _ in claves_repetidas.items():
    subset = df_std[(df_std['Empresa'] == empresa) & (df_std['Número de cargo'] == cargo)]
    distinct_rows = subset[target_cols].drop_duplicates()
    if len(distinct_rows) > 1:
        n_claves_repetidas_no_exactas += 1

print(f"  - de las cuales son duplicados EXACTOS en todas las columnas: {len(claves_repetidas) - n_claves_repetidas_no_exactas}")
print(f"  - de las cuales tienen ATRIBUTOS DIFERENTES entre filas (requieren revisión de negocio): {n_claves_repetidas_no_exactas}")


In [ ]:
# Hallazgo específico: contenido idéntico entre archivos con nombre de empresa distinto.
# Se reporta de forma agregada (sin exponer filas) para no asumir ni corregir en automático.
origenes_por_archivo = df_std.groupby('__archivo_origen__').size()
empresas_por_archivo = df_all.groupby('__archivo_origen__')['Empresa'].agg(lambda s: sorted(set(str(x) for x in s.dropna())))

print("Códigos de 'Empresa' presentes en cada archivo fuente (por nombre de archivo):")
for archivo, codigos in empresas_por_archivo.items():
    print(f"  - {archivo}: {codigos}")

# Comparación de contenido entre pares de archivos con distinto nombre de empresa
print()
print("Verificación de solapamiento de contenido entre archivos (mismas 8 columnas de negocio, ignorando el archivo de origen):")
archivos_lista = list(df_std['__archivo_origen__'].unique())
for i in range(len(archivos_lista)):
    for j in range(i + 1, len(archivos_lista)):
        a, b = archivos_lista[i], archivos_lista[j]
        set_a = set(map(tuple, df_std[df_std['__archivo_origen__'] == a][target_cols].astype(str).values))
        set_b = set(map(tuple, df_std[df_std['__archivo_origen__'] == b][target_cols].astype(str).values))
        overlap = set_a & set_b
        if overlap:
            pct = 100 * len(overlap) / max(len(set_a), 1)
            print(f"  ALERTA: '{a}' y '{b}' comparten {len(overlap)} filas con contenido 100% idéntico ({pct:.0f}% del contenido de '{a}').")


## 6. Construcción de la tabla final (sin dedup, sin enriquecimiento)

In [ ]:
df_final = df_std.copy()
df_final["Dependencia"] = None
df_final["Área"] = None
df_final["Fecha de actualización"] = FECHA_ACTUALIZACION  # None en esta fase

df_final = df_final.drop(columns=["__archivo_origen__"])
df_final = df_final[FINAL_COLUMNS]

print(f"Filas en la tabla final (sin deduplicar, tal como llegan de las 8 fuentes): {len(df_final)}")
print(f"Columnas finales: {list(df_final.columns)}")
print()
print("Nota: no se eliminó ningún registro. Los duplicados identificados en la sección 5")
print("quedan visibles en esta tabla para revisión y decisión posterior del usuario.")


## 7. QA final obligatorio

Métricas agregadas exigidas antes de declarar la entrega como lista.


In [ ]:
qa = {}
qa["archivos_esperados"] = EXPECTED_SOURCE_FILES
qa["archivos_encontrados"] = len(xlsx_files)
qa["filas_por_archivo"] = filas_por_archivo
qa["filas_consolidadas_antes_dedup"] = len(df_all)
qa["duplicados_exactos_filas"] = n_dup_exact_rows
qa["duplicados_exactos_grupos"] = n_dup_exact_groups
qa["claves_empresa_cargo_repetidas"] = int(len(claves_repetidas))
qa["claves_empresa_cargo_repetidas_con_atributos_distintos"] = n_claves_repetidas_no_exactas
qa["cargos_unicos_empresa_cargo"] = n_empresa_cargo_unico
qa["cargos_codigo_vacio"] = int(df_final["Número de cargo"].isna().sum())
qa["empresas_vacias"] = int(df_final["Empresa"].isna().sum())
qa["fechas_creacion_invalidas"] = int((~fecha_valida_mask).sum())
qa["fechas_creacion_vacias"] = int(fecha_vacia_mask.sum())
qa["numero_de_cargos_invalido"] = int((~n_cargos_valido_mask).sum())
qa["cargos_ocupados_invalido"] = int((~cargos_ocup_valido_mask).sum())

ambos_numericos = df_final["Número de cargos"].notna() & df_final["Cargos ocupados"].notna()
excede_mask = ambos_numericos & (df_final["Cargos ocupados"] > df_final["Número de cargos"])
qa["casos_cargos_ocupados_mayor_que_numero_de_cargos"] = int(excede_mask.sum())

qa["dependencia_vacia_filas"] = int(df_final["Dependencia"].isna().sum())
qa["area_vacia_filas"] = int(df_final["Área"].isna().sum())
qa["fecha_actualizacion_vacia_filas"] = int(df_final["Fecha de actualización"].isna().sum())

print("=== RESULTADOS QA ===")
for k, v in qa.items():
    print(f"{k}: {v}")

assert qa["dependencia_vacia_filas"] == len(df_final), "Dependencia debía quedar 100% vacía en esta fase."
assert qa["area_vacia_filas"] == len(df_final), "Área debía quedar 100% vacía en esta fase."
assert qa["fecha_actualizacion_vacia_filas"] == len(df_final), "Fecha de actualización debía quedar 100% vacía en esta fase."
print()
print("OK: Dependencia, Área y Fecha de actualización quedaron 100% vacías, como corresponde a esta fase.")


## 8. Tabla final de la Fase 1 (en memoria — sin exportar Excel intermedio)

`df_final` es el DataFrame que la Fase 2 enriquece y exporta como hoja
`Depuracion_Cargos` del único Excel del pipeline (sección 18). Aquí solo se
confirma su forma final, sin escribir ningún archivo.


In [ ]:
SHEET_NAME = "Depuracion_Cargos"

print(f"df_final listo en memoria: {len(df_final)} filas, columnas: {list(df_final.columns)}")


## 9. Resumen de resultados — Fase 1

In [ ]:
print("=" * 80)
print("RESUMEN DE EJECUCIÓN — Fase 1 (tabla base Kactus, en memoria)")
print("=" * 80)
print(f"Archivos fuente procesados: {len(xlsx_files)} de {EXPECTED_SOURCE_FILES} esperados")
print(f"Filas consolidadas (sin deduplicar): {len(df_final)}")
print(f"Clave candidata de granularidad: Empresa + Número de cargo")
print(f"  - únicos por esa clave: {n_empresa_cargo_unico} de {len(df_final)} filas")
print(f"Duplicados exactos (8 columnas de negocio): {n_dup_exact_rows} filas en {n_dup_exact_groups} grupos")
print(f"Claves Empresa+Cargo repetidas con atributos distintos: {n_claves_repetidas_no_exactas}")
print(f"Dependencia / Área / Fecha de actualización: vacías en esta fase (se completan en la Fase 2)")
print(f"Esta tabla se exporta como hoja 'Depuracion_Cargos' en {TABLA_FINAL_XLSX} al cierre de la Fase 2 (sección 18).")


# Fase 2 — Enriquecimiento con Consolidado 2025.xlsx

## Objetivo

Completar `Dependencia` y `Área` en la tabla base Kactus (2 004 cargos,
`Empresa + Número de cargo` 100% único) usando `Consolidado 2025.xlsx`,
tomando como referencia julio de 2026.

## Reglas de negocio validadas empíricamente en esta fase (ver evidencia abajo)

1. **Mapeo de Empresa — RESUELTO mediante reglas de negocio confirmadas por
   el usuario.** `GRUPO EMPRESA` es una etiqueta de agrupación
   corporativa/reporting y NO equivale a la empresa legal Kactus.
   `Nombre Empresa` sí permite reconstruir la empresa legal Kactus, mediante
   un catálogo de 11 reglas confirmadas (ver sección 12) que cubre las 8
   empresas Kactus, incluyendo el grupo `GRUPO EMPRESA = 'HABITEL HOTELS'`
   (`HABITEL SELECT`/`HABITEL PRIME`/`HABITEL NÓMINA COMPARTIDA` → Empresa 6;
   `OPERADORA` → Empresa 10; `LEMCO SALVIO` → Empresa 7, mismo origen legal
   que `LEMCO`). Se valida más abajo que estas reglas cubren el 100% de las
   332 filas de julio 2026 de ese grupo, sin necesidad de inferir ningún caso
   adicional.

2. **Conflictos Dependencia/Área.** Para las empresas resueltas, un mismo
   `Empresa + Cargo` puede tener múltiples combinaciones de `Dependencia`/
   `Área` en julio 2026 (el `Cargo` de Kactus es una plantilla de rol, no un
   puesto departamental único — confirmado también por la hoja auxiliar
   `Tbls_Dependencia_Area` del propio archivo, que ya marca este mismo
   problema con columnas `Revisar Dependencia`/`Revisar Área`). Estos casos
   NO se resuelven automáticamente: se documentan en la hoja
   `Conflictos_Dependencia_Area` del Excel final y quedan sin
   `Dependencia`/`Área` en la hoja principal.

3. No se modifican `Número de cargos`, `Cargos ocupados` ni ningún otro campo
   ya validado en la Fase 1. Los 68 casos `Cargos ocupados > Número de
   cargos` se preservan sin cambios.


## 10. Localización y auditoría de Consolidado 2025.xlsx

In [ ]:
CONSOLIDADO_CANDIDATES = list(REPO_ROOT.glob("Data/**/Consolidado*2025*.xlsx"))
print("Candidatos encontrados para 'Consolidado 2025.xlsx' dentro de Data/:")
for c in CONSOLIDADO_CANDIDATES:
    print(f"  - {c.relative_to(REPO_ROOT)}")

assert len(CONSOLIDADO_CANDIDATES) == 1, "Se esperaba exactamente 1 archivo Consolidado 2025.xlsx; revisar manualmente."
CONSOLIDADO_PATH = CONSOLIDADO_CANDIDATES[0]
print()
print(f"Archivo utilizado: {CONSOLIDADO_PATH.relative_to(REPO_ROOT)}")

wb_cons = openpyxl.load_workbook(CONSOLIDADO_PATH, read_only=True, data_only=True)
print(f"Hojas disponibles: {wb_cons.sheetnames}")

SHEET_CONSOLIDADO = "Consolidado2025"
assert SHEET_CONSOLIDADO in wb_cons.sheetnames, f"No se encontro la hoja {SHEET_CONSOLIDADO!r}"
ws_cons = wb_cons[SHEET_CONSOLIDADO]
print(f"Hoja utilizada: {SHEET_CONSOLIDADO!r} | max_row={ws_cons.max_row} max_col={ws_cons.max_column}")

CONS_HEADER = list(next(ws_cons.iter_rows(min_row=1, max_row=1, values_only=True)))
CONS_IDX = {name: i for i, name in enumerate(CONS_HEADER)}
print(f"Columnas disponibles ({len(CONS_HEADER)}): {CONS_HEADER}")

REQUIRED_CONS_COLUMNS = ["GRUPO EMPRESA", "Nombre Empresa", "COD. CARGO", "CARGO", "DEPENDENCIA", "AREA", "MES", "AÑO"]
faltantes_cons = [c for c in REQUIRED_CONS_COLUMNS if c not in CONS_IDX]
assert not faltantes_cons, f"Faltan columnas requeridas en Consolidado2025: {faltantes_cons}"
print()
print("Campos utilizados: Empresa <- 'Nombre Empresa' (validado, ver mapeo abajo); Cargo <- 'COD. CARGO';")
print("Dependencia <- 'DEPENDENCIA'; Área <- 'AREA'; Periodo <- 'MES' + 'AÑO'.")


## 11. Carga de filas y validación del periodo julio 2026

In [ ]:
CONS_ROWS = list(ws_cons.iter_rows(min_row=2, values_only=True))
wb_cons.close()

print(f"Filas de datos en Consolidado2025: {len(CONS_ROWS)}")

periodos = set()
for row in CONS_ROWS:
    periodos.add((row[CONS_IDX['AÑO']], row[CONS_IDX['MES']]))

def periodo_sort_key(p):
    anio, mes = p
    mes_num = int(str(mes).split('.')[0]) if mes else 0
    return (anio if anio is not None else -1, mes_num)

periodos_ordenados = sorted(periodos, key=periodo_sort_key)
periodo_min = periodos_ordenados[0]
periodo_max = periodos_ordenados[-1]

print(f"Periodo minimo detectado: AÑO={periodo_min[0]!r} MES={periodo_min[1]!r}")
print(f"Periodo maximo detectado: AÑO={periodo_max[0]!r} MES={periodo_max[1]!r}")

PERIODO_AUTORIZADO = (2026, "07.Julio")
existe_periodo_autorizado = PERIODO_AUTORIZADO in periodos
print(f"¿Existe el periodo autorizado {PERIODO_AUTORIZADO}?: {existe_periodo_autorizado}")
assert existe_periodo_autorizado, "El periodo julio 2026 no existe en Consolidado2025 — DETENER."

if periodo_max != PERIODO_AUTORIZADO:
    print()
    print(f"ALERTA: el periodo maximo detectado ({periodo_max}) es POSTERIOR al periodo autorizado {PERIODO_AUTORIZADO}.")
    print("No se cambia el periodo autorizado automaticamente. Reportar al usuario antes de usar un periodo distinto.")
else:
    print()
    print(f"CONFIRMADO: julio 2026 es el periodo maximo disponible en Consolidado2025. Se usa como periodo autorizado.")

rows_julio = [row for row in CONS_ROWS if (row[CONS_IDX['AÑO']], row[CONS_IDX['MES']]) == PERIODO_AUTORIZADO]
print(f"Filas de julio 2026: {len(rows_julio)}")


## 12. Mapeo Empresa (Consolidado) → Empresa (Kactus) — reglas de negocio confirmadas por el usuario

`GRUPO EMPRESA` es una agrupación corporativa/reporting y NO representa la
empresa legal/origen Kactus. `Nombre Empresa` es el resultado de reglas
manuales aplicadas durante la preparación mensual del cierre; para 6 de las 8
empresas es una relación 1 a 1 directa, pero para el grupo
`GRUPO EMPRESA = 'HABITEL HOTELS'` agrupa etiquetas de 3 empresas legales
Kactus distintas (6, 7 y 10) según el siguiente catálogo confirmado por el
usuario:

| Nombre Empresa (Consolidado) | Empresa Kactus | Archivo Kactus origen |
|---|---|---|
| `CHALLENGER` | 1 | `CHALLENGER SAS.xlsx` |
| `FUNDACIÓN CHALLENGER` | 5 | `FUNDACION CHALLENGER.xlsx` |
| `SKY FORWARDER` | 3 | `SKY FORWARDER SAS.xlsx` |
| `SKY INDUSTRIAL` | 8 | `SKY INDUSTRIAL.xlsx` |
| `SKY LOGÍSTICA INTEGRAL` | 9 | `SKY LOGISTICA INTEGRAL.xlsx` |
| `LEMCO` | 7 | `LEMCO SAS.xlsx` |
| `LEMCO SALVIO` | 7 | `LEMCO SAS.xlsx` (mismo origen legal; `GRUPO EMPRESA='HABITEL HOTELS'` en este caso es solo agrupación de reporting) |
| `OPERADORA` | 10 | `OPERADORA HABITEL SAS.xlsx` |
| `HABITEL PRIME` | 6 | `HABITEL SAS.xlsx` (tipo de nómina, ver catálogo en `Reglas_Normalizacion_Empresa_Contratos_Kactus.md`) |
| `HABITEL SELECT` | 6 | `HABITEL SAS.xlsx` (idem) |
| `HABITEL NÓMINA COMPARTIDA` | 6 | `HABITEL SAS.xlsx` (idem) |

El mapeo se valida empíricamente comparando, para cada `Nombre Empresa`, el
porcentaje de códigos `COD. CARGO` de julio 2026 que existen en el catálogo
`Cargo` de la empresa Kactus correspondiente. Se documenta cualquier
`Nombre Empresa` de julio 2026 que NO esté cubierto por esta tabla — esos
casos no se infieren, quedan pendientes de decisión del usuario.


In [ ]:
NOMBRE_EMPRESA_A_EMPRESA_KACTUS = {
    "CHALLENGER": "1",
    "FUNDACIÓN CHALLENGER": "5",
    "SKY FORWARDER": "3",
    "SKY INDUSTRIAL": "8",
    "SKY LOGÍSTICA INTEGRAL": "9",
    "LEMCO": "7",
    "LEMCO SALVIO": "7",
    "OPERADORA": "10",
    "HABITEL PRIME": "6",
    "HABITEL SELECT": "6",
    "HABITEL NÓMINA COMPARTIDA": "6",
}

EMPRESAS_KACTUS_CONCILIABLES = set(NOMBRE_EMPRESA_A_EMPRESA_KACTUS.values())
print(f"Empresas Kactus cubiertas por reglas confirmadas: {sorted(EMPRESAS_KACTUS_CONCILIABLES)}")
print(f"Empresas Kactus existentes en la tabla base: {sorted(df_final['Empresa'].unique())}")

# Validación empírica del mapeo: solapamiento de COD. CARGO (julio 2026) contra el catálogo Cargo de Kactus.
cargo_sets_kactus = {}
for empresa_code in EMPRESAS_KACTUS_CONCILIABLES:
    cargo_sets_kactus[empresa_code] = set(df_final.loc[df_final["Empresa"] == empresa_code, "Número de cargo"].dropna())

por_nombre_empresa_codigos = {}
for row in rows_julio:
    ne = row[CONS_IDX["Nombre Empresa"]]
    cc = row[CONS_IDX["COD. CARGO"]]
    if cc is None:
        continue
    por_nombre_empresa_codigos.setdefault(ne, set()).add(str(cc).strip())

print()
print("Validación de solapamiento (julio 2026) por Nombre Empresa mapeado:")
for ne, empresa_code in NOMBRE_EMPRESA_A_EMPRESA_KACTUS.items():
    codigos = por_nombre_empresa_codigos.get(ne, set())
    kset = cargo_sets_kactus[empresa_code]
    overlap = codigos & kset
    pct = 100 * len(overlap) / len(codigos) if codigos else 0
    print(f"  Nombre Empresa={ne!r} -> Empresa Kactus={empresa_code}: {len(overlap)}/{len(codigos)} codigos coinciden ({pct:.1f}%)")

nombre_empresa_no_mapeados = sorted({
    (row[CONS_IDX['GRUPO EMPRESA']], row[CONS_IDX['Nombre Empresa']])
    for row in rows_julio
    if row[CONS_IDX['Nombre Empresa']] not in NOMBRE_EMPRESA_A_EMPRESA_KACTUS
}, key=lambda x: str(x))

print()
print(f"Combinaciones GRUPO EMPRESA / Nombre Empresa de julio 2026 NO cubiertas por las reglas confirmadas: {len(nombre_empresa_no_mapeados)}")
for ge, ne in nombre_empresa_no_mapeados:
    print(f"  GRUPO EMPRESA={ge!r}  Nombre Empresa={ne!r}  <- NO se infiere, pendiente de decisión del usuario")


## 13. Construcción del lookup Empresa+Cargo → Dependencia/Área y validación de cardinalidad

In [ ]:
key_to_pairs = {}
key_to_count = {}
filas_resueltas = 0
filas_no_resueltas_empresa = 0

for row in rows_julio:
    ne = row[CONS_IDX["Nombre Empresa"]]
    empresa_kactus = NOMBRE_EMPRESA_A_EMPRESA_KACTUS.get(ne)
    if empresa_kactus is None:
        filas_no_resueltas_empresa += 1
        continue
    filas_resueltas += 1
    cod = row[CONS_IDX["COD. CARGO"]]
    if cod is None:
        continue
    cod = str(cod).strip()
    dep = row[CONS_IDX["DEPENDENCIA"]]
    area = row[CONS_IDX["AREA"]]
    dep = dep.strip() if isinstance(dep, str) else dep
    area = area.strip() if isinstance(area, str) else area
    key = (empresa_kactus, cod)
    key_to_pairs.setdefault(key, set()).add((dep, area))
    key_to_count[key] = key_to_count.get(key, 0) + 1  # 1 fila = 1 colaborador activo en el cierre julio 2026

print(f"Filas julio 2026 con Empresa resuelta (mapeable a Kactus): {filas_resueltas}")
print(f"Filas julio 2026 SIN resolver (Nombre Empresa fuera de las reglas confirmadas): {filas_no_resueltas_empresa}")
print(f"Claves unicas Empresa+Cargo con al menos 1 fila resuelta: {len(key_to_pairs)}")

claves_conflicto = {k: v for k, v in key_to_pairs.items() if len(v) > 1}
claves_seguras = {k: v for k, v in key_to_pairs.items() if len(v) == 1}

print(f"Claves SIN conflicto (una sola combinacion Dependencia/Area): {len(claves_seguras)}")
print(f"Claves CON conflicto (multiples combinaciones Dependencia/Area): {len(claves_conflicto)}")
if claves_conflicto:
    n_combos = [len(v) for v in claves_conflicto.values()]
    print(f"  combinaciones distintas por clave conflictiva: minimo={min(n_combos)} maximo={max(n_combos)} promedio={sum(n_combos)/len(n_combos):.1f}")

LOOKUP_SEGURO = {k: next(iter(v)) for k, v in claves_seguras.items()}


## 13b. Resolución de HABITEL / OPERADORA / LEMCO SALVIO — regla de negocio confirmada

**Historial:** se auditaron 19 columnas técnicas de `Consolidado2025` (`COD`,
`CCO`, `CARGO_CCO`, `TIPO_CONTR`, `AGRUPADOR`, `DEPARTAMENTO`,
`DEPENDENCIA_PATRON`, `AREA_PATRON`, etc.) buscando un campo estable e
independiente de `Nombre Empresa` que preservara la empresa legal Kactus de
origen para las 332 filas de julio 2026 con `GRUPO EMPRESA = 'HABITEL
HOTELS'`. Ninguna columna técnica cumplió ese rol (`COD` es una concatenación
de `Nombre Empresa` + `Dependencia`; `CCO` reutiliza numeración entre
`HABITEL SELECT`/`HABITEL PRIME` y usa un esquema distinto para
`LEMCO SALVIO`).

**Resolución:** el usuario confirmó que la propia granularidad de
`Nombre Empresa` (no `GRUPO EMPRESA`) ya distingue la empresa legal de
origen, mediante reglas de negocio conocidas por Planeación de Personal (ver
tabla de la sección 12 y `Reglas_Normalizacion_Empresa_Contratos_Kactus.md`):

- `OPERADORA` → Empresa Kactus 10 (`OPERADORA HABITEL SAS.xlsx`).
- `LEMCO SALVIO` → Empresa Kactus 7 (`LEMCO SAS.xlsx`) — mismo origen legal
  que `LEMCO`; `GRUPO EMPRESA='HABITEL HOTELS'` es aquí solo una etiqueta de
  reporting de contratos, no la empresa legal.
- `HABITEL PRIME`, `HABITEL SELECT`, `HABITEL NÓMINA COMPARTIDA` → Empresa
  Kactus 6 (`HABITEL SAS.xlsx`) — corresponden a distintos "Tipo de Nómina"
  dentro de la misma empresa legal (ver catálogo Tipo de Nómina en el
  Markdown de reglas).

Se valida a continuación que estas 3 reglas cubren efectivamente el 100% de
las filas de julio 2026 con `GRUPO EMPRESA='HABITEL HOTELS'`, sin necesidad
de inferir ningún caso adicional.


In [ ]:
habitel_hotels_julio = [row for row in rows_julio if row[CONS_IDX["GRUPO EMPRESA"]] == "HABITEL HOTELS"]
habitel_hotels_nombre_empresa = {}
for row in habitel_hotels_julio:
    ne = row[CONS_IDX["Nombre Empresa"]]
    habitel_hotels_nombre_empresa[ne] = habitel_hotels_nombre_empresa.get(ne, 0) + 1

print(f"Filas julio 2026 con GRUPO EMPRESA='HABITEL HOTELS': {len(habitel_hotels_julio)}")
print("Distribución por Nombre Empresa:")
cubiertas = 0
no_cubiertas = 0
for ne, cnt in sorted(habitel_hotels_nombre_empresa.items(), key=lambda x: str(x[0])):
    empresa_kactus = NOMBRE_EMPRESA_A_EMPRESA_KACTUS.get(ne)
    estado = f"-> Empresa Kactus {empresa_kactus}" if empresa_kactus else "-> SIN REGLA CONFIRMADA"
    print(f"  {ne!r}: {cnt} filas {estado}")
    if empresa_kactus:
        cubiertas += cnt
    else:
        no_cubiertas += cnt

print()
print(f"Filas cubiertas por regla confirmada: {cubiertas} / {len(habitel_hotels_julio)} ({100*cubiertas/len(habitel_hotels_julio):.1f}%)")
print(f"Filas SIN cubrir (requieren decisión del usuario, no se infieren): {no_cubiertas}")

RESULTADO_BUSQUEDA_IDENTIFICADOR_LEGAL = (
    "Ninguna columna técnica de Consolidado2025 preserva un identificador legal de empresa "
    "independiente de 'Nombre Empresa' (se evaluaron 19 columnas candidatas). La ambigüedad se "
    f"resolvió mediante regla de negocio confirmada por el usuario sobre 'Nombre Empresa': "
    f"cubre {cubiertas} de {len(habitel_hotels_julio)} filas de GRUPO EMPRESA='HABITEL HOTELS' "
    f"en julio 2026 ({100*cubiertas/len(habitel_hotels_julio):.1f}%)."
)
print()
print(RESULTADO_BUSQUEDA_IDENTIFICADOR_LEGAL)

assert no_cubiertas == 0, "Hay filas de HABITEL HOTELS julio 2026 sin regla confirmada — DETENER y reportar al usuario."


## 14. Evidencia de conflictos — hoja `Conflictos_Dependencia_Area`

`claves_conflicto` son las claves `Empresa+Cargo` del **lookup de Consolidado
julio 2026** (sección 13) con más de una combinación Dependencia/Área. Esto
es distinto de "cargos Kactus con Dependencia/Área múltiple" (sección 15):
una clave del lookup puede no tener ningún cargo Kactus correspondiente (por
ejemplo, un `COD. CARGO` que existe en Consolidado pero no en el catálogo
Kactus de esa empresa). Se marca explícitamente con la columna
`Existe en catálogo Kactus` (Sí/No) para que esa diferencia quede
trazable — no se descartan las claves sin match, se documentan igual.

Este DataFrame se mantiene en memoria y se exporta como tercera hoja
(`Conflictos_Dependencia_Area`) del Excel final en la sección 18 — no genera
un archivo Excel independiente.


In [ ]:
conflict_rows = []
for (empresa_code, cargo_code), pares in claves_conflicto.items():
    kactus_match = df_final[(df_final["Empresa"] == empresa_code) & (df_final["Número de cargo"] == cargo_code)]
    existe_en_kactus = len(kactus_match) > 0
    nombre_cargo = kactus_match["Nombre del cargo"].iloc[0] if existe_en_kactus else None
    nombre_empresa_kactus = kactus_match["Nombre Empresa"].iloc[0] if existe_en_kactus else None
    for dep, area in sorted(pares, key=lambda x: (str(x[0]), str(x[1]))):
        conflict_rows.append({
            "Empresa": empresa_code,
            "Nombre Empresa": nombre_empresa_kactus,
            "Número de cargo": cargo_code,
            "Nombre del cargo": nombre_cargo,
            "Existe en catálogo Kactus": "Sí" if existe_en_kactus else "No",
            "Dependencia (opción encontrada)": dep,
            "Área (opción encontrada)": area,
            "Combinaciones distintas para esta clave": len(pares),
        })

CONFLICTOS_COLS = [
    "Empresa", "Nombre Empresa", "Número de cargo", "Nombre del cargo",
    "Existe en catálogo Kactus", "Dependencia (opción encontrada)",
    "Área (opción encontrada)", "Combinaciones distintas para esta clave",
]
df_conflictos = pd.DataFrame(conflict_rows, columns=CONFLICTOS_COLS)
if len(df_conflictos) > 0:
    df_conflictos = df_conflictos.sort_values(
        ["Existe en catálogo Kactus", "Combinaciones distintas para esta clave", "Empresa", "Número de cargo"],
        ascending=[False, False, True, True],
    )

n_claves_conflicto_total = len(claves_conflicto)
if len(df_conflictos) > 0:
    n_claves_conflicto_con_kactus = int((df_conflictos.groupby(["Empresa", "Número de cargo"])["Existe en catálogo Kactus"].first() == "Sí").sum())
else:
    n_claves_conflicto_con_kactus = 0
n_claves_conflicto_sin_kactus = n_claves_conflicto_total - n_claves_conflicto_con_kactus

print(f"Claves conflictivas totales del lookup Consolidado julio 2026: {n_claves_conflicto_total}")
print(f"  Con correspondencia en el catálogo Kactus (cuentan como cargo Kactus 'Dependencia/Área múltiple'): {n_claves_conflicto_con_kactus}")
print(f"  SIN correspondencia en el catálogo Kactus (clave de Consolidado sin cargo Kactus asociado): {n_claves_conflicto_sin_kactus}")
print(f"Filas totales en la evidencia de conflictos (una fila por combinación Dependencia/Área): {len(df_conflictos)}")


## 15. Reclasificación y enriquecimiento (Dependencia / Área)

Estado de conciliación por cargo (4 categorías, mutuamente excluyentes):

- **Empresa no conciliable automáticamente**: `Empresa` Kactus del cargo no
  está cubierta por ninguna regla confirmada de la sección 12 (con las
  reglas actuales, cubre las 8 empresas Kactus — ver validación abajo).
- **Dependencia/Área múltiple**: la clave `Empresa+Cargo` tiene presencia en
  el cierre julio 2026, pero con más de una combinación Dependencia/Área —
  no se elige arbitrariamente, queda como evidencia de depuración.
- **Con presencia en cierre julio-2026**: clave con exactamente una
  combinación Dependencia/Área — se completa.
- **Sin presencia en cierre julio-2026**: el cargo no tiene ningún
  colaborador activo en el cierre de julio 2026. NO implica que deba
  eliminarse o inactivarse — es insumo de depuración (cargo vacante,
  disponible o pendiente de revisión).

`Ind. Actividad` se conserva exactamente como llega de Kactus — no se
sobrescribe con ninguna señal de Consolidado.


In [ ]:
def lookup_dependencia(row):
    key = (row["Empresa"], row["Número de cargo"])
    par = LOOKUP_SEGURO.get(key)
    return par[0] if par else None

def lookup_area(row):
    key = (row["Empresa"], row["Número de cargo"])
    par = LOOKUP_SEGURO.get(key)
    return par[1] if par else None

df_enriquecido = df_final.copy()
df_enriquecido["Dependencia"] = df_enriquecido.apply(lookup_dependencia, axis=1)
df_enriquecido["Área"] = df_enriquecido.apply(lookup_area, axis=1)

keys_conflicto_set = set(claves_conflicto.keys())
empresa_no_conciliable_mask = ~df_enriquecido["Empresa"].isin(EMPRESAS_KACTUS_CONCILIABLES)
print(f"Cargos con Empresa Kactus fuera de las reglas confirmadas: {int(empresa_no_conciliable_mask.sum())}")

def estado_conciliacion(row):
    key = (row["Empresa"], row["Número de cargo"])
    if row["Empresa"] not in EMPRESAS_KACTUS_CONCILIABLES:
        return "Empresa no conciliable automáticamente"
    if key in keys_conflicto_set:
        return "Dependencia/Área múltiple"
    if key in LOOKUP_SEGURO:
        return "Con presencia en cierre julio-2026"
    return "Sin presencia en cierre julio-2026"

df_enriquecido["__estado_conciliacion__"] = df_enriquecido.apply(estado_conciliacion, axis=1)

def colaboradores_activos(row):
    if row["Empresa"] not in EMPRESAS_KACTUS_CONCILIABLES:
        return None  # empresa no conciliable: no se puede determinar
    key = (row["Empresa"], row["Número de cargo"])
    return key_to_count.get(key, 0)

df_enriquecido["__colaboradores_activos_julio2026__"] = df_enriquecido.apply(colaboradores_activos, axis=1)

def n_dependencias_distintas(row):
    key = (row["Empresa"], row["Número de cargo"])
    pares = key_to_pairs.get(key)
    if not pares:
        return 0
    return len({p[0] for p in pares})

def n_areas_distintas(row):
    key = (row["Empresa"], row["Número de cargo"])
    pares = key_to_pairs.get(key)
    if not pares:
        return 0
    return len({p[1] for p in pares})

df_enriquecido["__n_dependencias_distintas__"] = df_enriquecido.apply(n_dependencias_distintas, axis=1)
df_enriquecido["__n_areas_distintas__"] = df_enriquecido.apply(n_areas_distintas, axis=1)

n_total_cargos = len(df_enriquecido)
n_con_dependencia = int(df_enriquecido["Dependencia"].notna().sum())
n_con_area = int(df_enriquecido["Área"].notna().sum())
estado_counts = df_enriquecido["__estado_conciliacion__"].value_counts()

print(f"Cargos totales: {n_total_cargos}")
print()
print("Distribución por Estado de conciliación:")
for estado, cnt in estado_counts.items():
    print(f"  {estado}: {cnt} ({100*cnt/n_total_cargos:.1f}%)")
print()
print(f"Cobertura final Dependencia/Área (cargos con valor asignado): {n_con_dependencia} ({100*n_con_dependencia/n_total_cargos:.1f}%)")


## 15b. Conciliación QA: Ind. Actividad (Kactus) vs Presencia en el cierre julio-2026

Cruce restringido a los cargos con Empresa Kactus cubierta por una regla
confirmada (sección 12) — con las reglas actuales, cubre las 8 empresas
Kactus, por lo que en la práctica incluye los 2.004 cargos. `Ind. Actividad
!= A` no está completamente actualizado en Kactus — este cruce es
precisamente uno de los resultados centrales de la depuración, no un simple
chequeo de calidad.


In [ ]:
df_reconciliables = df_enriquecido[~empresa_no_conciliable_mask].copy()
df_reconciliables["__presente__"] = df_reconciliables["__estado_conciliacion__"].isin(
    ["Con presencia en cierre julio-2026", "Dependencia/Área múltiple"]
)
df_reconciliables["__ind_actividad_a__"] = df_reconciliables["Ind. Actividad"] == "A"

cross = pd.crosstab(df_reconciliables["__ind_actividad_a__"], df_reconciliables["__presente__"])

n_a_presente = int(((df_reconciliables["__ind_actividad_a__"]) & (df_reconciliables["__presente__"])).sum())
n_a_ausente = int(((df_reconciliables["__ind_actividad_a__"]) & (~df_reconciliables["__presente__"])).sum())
n_noa_presente = int(((~df_reconciliables["__ind_actividad_a__"]) & (df_reconciliables["__presente__"])).sum())
n_noa_ausente = int(((~df_reconciliables["__ind_actividad_a__"]) & (~df_reconciliables["__presente__"])).sum())

print(f"Cargos evaluados (empresas reconciliables): {len(df_reconciliables)}")
print(f"  Ind.Actividad=A  + presente en cierre julio2026:  {n_a_presente}")
print(f"  Ind.Actividad=A  + AUSENTE del cierre julio2026:  {n_a_ausente}   <- candidatos a revisar (activo en Kactus, sin colaborador en julio)")
print(f"  Ind.Actividad!=A + presente en cierre julio2026:  {n_noa_presente}   <- candidatos a revisar (inactivo/otro en Kactus, con colaborador en julio)")
print(f"  Ind.Actividad!=A + AUSENTE del cierre julio2026:  {n_noa_ausente}")
print()
print("Nota: 'ausente' no implica que el cargo deba inactivarse; es evidencia para la depuración.")


## 15c. Cargos ocupados (Kactus) vs Colaboradores activos (cierre julio-2026) — solo QA, no se sobrescribe

In [ ]:
df_reconciliables["__cargos_ocupados_num__"] = pd.to_numeric(df_reconciliables["Cargos ocupados"], errors="coerce")
comparable_mask = df_reconciliables["__cargos_ocupados_num__"].notna() & df_reconciliables["__colaboradores_activos_julio2026__"].notna()
discrepancia_mask = comparable_mask & (df_reconciliables["__cargos_ocupados_num__"] != df_reconciliables["__colaboradores_activos_julio2026__"])

n_comparables = int(comparable_mask.sum())
n_discrepancias = int(discrepancia_mask.sum())

print(f"Cargos comparables (Cargos ocupados Kactus vs Colaboradores activos julio2026): {n_comparables}")
print(f"Discrepancias encontradas: {n_discrepancias} ({100*n_discrepancias/n_comparables:.1f}% de los comparables)" if n_comparables else "Sin base comparable")
print("No se sobrescribe 'Cargos ocupados' en la tabla principal — solo se reporta como hallazgo de depuración.")


## 16. Fecha de actualización — se mantiene VACÍA

In [ ]:
# La tarea aún no está cerrada (ambigüedad Empresa Habitel/Operadora y 62 conflictos Dependencia/Área
# quedan documentados pero sin resolver). Por instrucción explícita del usuario, esta columna
# NO se rellena hasta que se autorice el cierre definitivo de toda la tarea.
FECHA_ACTUALIZACION = None
df_enriquecido["Fecha de actualización"] = pd.NaT
print(f"Fecha de actualización: VACÍA en las {len(df_enriquecido)} filas (tarea aún no cerrada, pendiente de autorización final).")


## 16b. Los 315 cargos previamente clasificados como "Empresa no conciliable" — recalculados

Antes de las reglas confirmadas de la sección 12, los cargos Kactus de
Empresa 6 (HABITEL S.A.S.) y 10 (OPERADORA HABITEL SAS) — 315 cargos en
total (156 + 159) — quedaban sin evaluar por ambigüedad de `GRUPO EMPRESA`.
Con las reglas confirmadas, `Empresa` ya no depende de un mapeo ambiguo:
viene directamente del archivo Kactus de origen. Se recalcula su nuevo
estado de conciliación.


In [ ]:
cargos_habitel_operadora = df_enriquecido[df_enriquecido["Empresa"].isin(["6", "10"])]
print(f"Cargos Kactus de Empresa 6 (HABITEL S.A.S.) o 10 (OPERADORA HABITEL SAS): {len(cargos_habitel_operadora)}")
print()
print("Nuevo estado de conciliación (antes: 100% 'Empresa no conciliable automáticamente'):")
for estado, cnt in cargos_habitel_operadora["__estado_conciliacion__"].value_counts().items():
    print(f"  {estado}: {cnt}")

print()
print("Desglose por Empresa Kactus:")
for empresa_code, label in [("6", "HABITEL S.A.S."), ("10", "OPERADORA HABITEL SAS")]:
    sub = cargos_habitel_operadora[cargos_habitel_operadora["Empresa"] == empresa_code]
    print(f"  Empresa {empresa_code} ({label}): {len(sub)} cargos")
    for estado, cnt in sub["__estado_conciliacion__"].value_counts().items():
        print(f"    {estado}: {cnt}")

still_ambiguous = int((cargos_habitel_operadora["__estado_conciliacion__"] == "Empresa no conciliable automáticamente").sum())
print()
print(f"Cargos que SIGUEN sin resolver tras las reglas confirmadas: {still_ambiguous} de {len(cargos_habitel_operadora)}")


## 17. QA final obligatorio — Fase 2

In [ ]:
qa2 = {}
qa2["cargos_kactus_esperados"] = 2004
qa2["cargos_kactus_finales"] = len(df_enriquecido)
qa2["filas_julio_2026_en_consolidado"] = len(rows_julio)
qa2["claves_unicas_lookup_julio_2026"] = len(key_to_pairs)
qa2["claves_lookup_sin_conflicto"] = len(claves_seguras)
qa2["claves_lookup_con_conflicto"] = len(claves_conflicto)

qa2["cargos_con_presencia_cierre_julio2026"] = int(estado_counts.get("Con presencia en cierre julio-2026", 0)) + int(estado_counts.get("Dependencia/Área múltiple", 0))
qa2["cargos_sin_presencia_cierre_julio2026"] = int(estado_counts.get("Sin presencia en cierre julio-2026", 0))
qa2["cargos_empresa_no_conciliable"] = int(estado_counts.get("Empresa no conciliable automáticamente", 0))
qa2["cargos_dependencia_area_multiple"] = int(estado_counts.get("Dependencia/Área múltiple", 0))
qa2["cargos_dependencia_area_unica"] = int(estado_counts.get("Con presencia en cierre julio-2026", 0))

qa2["ind_actividad_A_presente"] = n_a_presente
qa2["ind_actividad_A_ausente"] = n_a_ausente
qa2["ind_actividad_noA_presente"] = n_noa_presente
qa2["ind_actividad_noA_ausente"] = n_noa_ausente

qa2["cargos_ocupados_vs_activos_comparables"] = n_comparables
qa2["cargos_ocupados_vs_activos_discrepancias"] = n_discrepancias

qa2["cargos_con_dependencia"] = n_con_dependencia
qa2["cargos_sin_dependencia"] = n_total_cargos - n_con_dependencia
qa2["cobertura_enriquecimiento_pct"] = round(100 * n_con_dependencia / n_total_cargos, 1)

qa2["resultado_busqueda_identificador_legal_habitel_operadora"] = RESULTADO_BUSQUEDA_IDENTIFICADOR_LEGAL

# Preservación de hallazgos de Fase 1
excede_mask_fase2 = df_enriquecido["Número de cargos"].notna() & df_enriquecido["Cargos ocupados"].notna() & (df_enriquecido["Cargos ocupados"] > df_enriquecido["Número de cargos"])
qa2["casos_cargos_ocupados_mayor_que_numero_de_cargos_preservados"] = int(excede_mask_fase2.sum())
qa2["fechas_creacion_vacias_preservadas"] = int(df_enriquecido["Fecha de creación"].isna().sum())
qa2["ind_actividad_preservado_sin_modificar"] = bool((df_enriquecido["Ind. Actividad"].fillna("") == df_final["Ind. Actividad"].fillna("")).all())

dup_final_mask = df_enriquecido.duplicated(subset=["Empresa", "Número de cargo"], keep=False)
qa2["duplicados_finales_empresa_cargo"] = int(dup_final_mask.sum())

qa2["fecha_actualizacion_vacia"] = bool(df_enriquecido["Fecha de actualización"].isna().all())

qa2["filas_finales"] = len(df_enriquecido)

print("=== RESULTADOS QA — FASE 2 ===")
for k, v in qa2.items():
    print(f"{k}: {v}")

assert qa2["casos_cargos_ocupados_mayor_que_numero_de_cargos_preservados"] == 68, "El conteo de 68 casos QA de Fase 1 no se preservó — revisar."
assert qa2["duplicados_finales_empresa_cargo"] == 0, "Aparecieron duplicados Empresa+Cargo en la tabla final — DETENER."
assert qa2["ind_actividad_preservado_sin_modificar"], "Ind. Actividad fue modificado respecto a Kactus — DETENER."
assert qa2["fecha_actualizacion_vacia"], "Fecha de actualización debía quedar vacía — DETENER."
assert n_total_cargos == 2004, "Se perdieron o agregaron filas respecto a las 2004 esperadas — DETENER."
print()
print("OK: 68 casos QA de Fase 1 preservados. Ind. Actividad sin modificar. Fecha de actualización vacía.")
print("OK: cero duplicados Empresa+Cargo. 2004 filas preservadas.")


## 18. Exportación del Excel final — Tabla_Depuracion_Cargos.xlsx (único archivo del pipeline)

3 hojas, sin datos personales (todo a nivel de cargo, no de empleado):

- `Depuracion_Cargos`: exactamente las 11 columnas originales, sin columnas
  auxiliares.
- `QA_Conciliacion`: control por `Empresa + Número de cargo`.
- `Conflictos_Dependencia_Area`: evidencia de las claves conflictivas del
  lookup de julio 2026 (sección 14), incluyendo las que no tienen
  correspondencia en el catálogo Kactus.


In [ ]:
df_export_final = df_enriquecido[FINAL_COLUMNS].copy()

qa_conciliacion_cols = [
    "Empresa", "Número de cargo", "Ind. Actividad", "Cargos ocupados",
    "__presente_qa__", "__colaboradores_activos_julio2026__", "__estado_conciliacion__",
    "__n_dependencias_distintas__", "__n_areas_distintas__", "__observacion_qa__",
]

def presente_qa(row):
    if row["Empresa"] not in EMPRESAS_KACTUS_CONCILIABLES:
        return None
    return row["__estado_conciliacion__"] in ("Con presencia en cierre julio-2026", "Dependencia/Área múltiple")

def observacion_qa(row):
    estado = row["__estado_conciliacion__"]
    if estado == "Empresa no conciliable automáticamente":
        return "Empresa Kactus sin regla de negocio confirmada en Consolidado — no evaluado"
    if estado == "Dependencia/Área múltiple":
        return f"{row['__n_dependencias_distintas__']} Dependencia(s) / {row['__n_areas_distintas__']} Área(s) distintas en julio 2026 — ver hoja Conflictos_Dependencia_Area"
    if estado == "Sin presencia en cierre julio-2026":
        return "Sin colaborador activo en el cierre julio 2026 — no implica inactivar el cargo"
    ocup = row["Cargos ocupados"]
    activos = row["__colaboradores_activos_julio2026__"]
    if pd.notna(ocup) and activos is not None and int(ocup) != int(activos):
        return f"Discrepancia: Cargos ocupados Kactus={int(ocup)} vs Colaboradores activos julio2026={int(activos)}"
    return ""

df_enriquecido["__presente_qa__"] = df_enriquecido.apply(presente_qa, axis=1)
df_enriquecido["__observacion_qa__"] = df_enriquecido.apply(observacion_qa, axis=1)

df_qa = df_enriquecido[qa_conciliacion_cols].copy()
df_qa.columns = [
    "Empresa", "Número de cargo", "Ind. Actividad Kactus", "Cargos ocupados Kactus",
    "Presencia cierre julio 2026", "Colaboradores activos julio 2026", "Estado conciliación",
    "Número de Dependencias distintas", "Número de Áreas distintas", "Observación QA",
]

with pd.ExcelWriter(TABLA_FINAL_XLSX, engine="openpyxl") as writer:
    df_export_final.to_excel(writer, sheet_name=SHEET_NAME, index=False)
    df_qa.to_excel(writer, sheet_name="QA_Conciliacion", index=False)
    df_conflictos.to_excel(writer, sheet_name="Conflictos_Dependencia_Area", index=False)

wb_f = openpyxl.load_workbook(TABLA_FINAL_XLSX)

def formatear_hoja(ws_f, df_cols, table_name, date_cols=(), numeric_cols=()):
    for cell in ws_f[1]:
        cell.font = Font(bold=True)
    for col_name in date_cols:
        col_idx = df_cols.index(col_name) + 1
        letter = get_column_letter(col_idx)
        for row in range(2, ws_f.max_row + 1):
            ws_f[f"{letter}{row}"].number_format = "yyyy-mm-dd"
    for col_name in numeric_cols:
        col_idx = df_cols.index(col_name) + 1
        letter = get_column_letter(col_idx)
        for row in range(2, ws_f.max_row + 1):
            ws_f[f"{letter}{row}"].number_format = "0"
    n_rows_f = ws_f.max_row
    n_cols_f = ws_f.max_column
    last_col_f = get_column_letter(n_cols_f)
    tabla_f = Table(displayName=table_name, ref=f"A1:{last_col_f}{n_rows_f}")
    tabla_f.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showFirstColumn=False, showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    ws_f.add_table(tabla_f)
    ws_f.freeze_panes = "A2"
    for col_idx, col_name in enumerate(df_cols, start=1):
        letter = get_column_letter(col_idx)
        ws_f.column_dimensions[letter].width = min(max(len(str(col_name)) + 2, 12), 45)

ws_main = wb_f[SHEET_NAME]
formatear_hoja(ws_main, list(df_export_final.columns), "TablaDepuracionCargosFinal",
               date_cols=["Fecha de creación", "Fecha de actualización"],
               numeric_cols=["Número de cargos", "Cargos ocupados"])

ws_qa = wb_f["QA_Conciliacion"]
formatear_hoja(ws_qa, list(df_qa.columns), "TablaQAConciliacion",
               numeric_cols=["Cargos ocupados Kactus", "Colaboradores activos julio 2026", "Número de Dependencias distintas", "Número de Áreas distintas"])

ws_conf = wb_f["Conflictos_Dependencia_Area"]
formatear_hoja(ws_conf, list(df_conflictos.columns), "TablaConflictosDependenciaArea",
               numeric_cols=["Combinaciones distintas para esta clave"])

wb_f.save(TABLA_FINAL_XLSX)
print(f"Excel final generado: {TABLA_FINAL_XLSX}")
print(f"Hoja '{SHEET_NAME}': {len(df_export_final)} filas, {len(df_export_final.columns)} columnas (exactas, sin auxiliares)")
print(f"Hoja 'QA_Conciliacion': {len(df_qa)} filas, {len(df_qa.columns)} columnas")
print(f"Hoja 'Conflictos_Dependencia_Area': {len(df_conflictos)} filas, {len(df_conflictos.columns)} columnas")
print(f"  Claves conflictivas: {n_claves_conflicto_total} totales ({n_claves_conflicto_con_kactus} con match Kactus, {n_claves_conflicto_sin_kactus} sin match Kactus)")
print()
print("Este es el ÚNICO archivo Excel generado por el pipeline (3 hojas). No se generan archivos Excel intermedios.")


## 19. Resumen de resultados — Fase 2

In [ ]:
print("=" * 80)
print("RESUMEN DE EJECUCIÓN — Tabla_Depuracion_Cargos.xlsx (Fase 2)")
print("=" * 80)
print(f"Consolidado 2025.xlsx: {CONSOLIDADO_PATH.relative_to(REPO_ROOT)} | hoja: {SHEET_CONSOLIDADO}")
print(f"Periodo autorizado: julio 2026 | filas en ese periodo: {len(rows_julio)}")
print(f"Nombre Empresa mapeados de forma segura: {list(NOMBRE_EMPRESA_A_EMPRESA_KACTUS.keys())}")
print(f"Empresas Kactus cubiertas: {sorted(EMPRESAS_KACTUS_CONCILIABLES)} (8 de 8 empresas del catálogo Kactus)")
print(f"Cargos Empresa 6/10 (HABITEL/OPERADORA) que siguen sin resolver: {still_ambiguous}")
print(f"Cargos totales: {n_total_cargos}")
print(f"  Con presencia en cierre julio-2026:      {qa2['cargos_con_presencia_cierre_julio2026']}")
print(f"  Sin presencia en cierre julio-2026:      {qa2['cargos_sin_presencia_cierre_julio2026']}")
print(f"  Empresa no conciliable automáticamente:  {qa2['cargos_empresa_no_conciliable']}")
print(f"  Dependencia/Área múltiple:               {qa2['cargos_dependencia_area_multiple']}")
print(f"Cobertura final de Dependencia/Área: {qa2['cobertura_enriquecimiento_pct']}%")
print(f"Claves con conflicto Dependencia/Área (lookup Consolidado): {n_claves_conflicto_total} totales — {n_claves_conflicto_con_kactus} con match Kactus (= {qa2['cargos_dependencia_area_multiple']} cargos), {n_claves_conflicto_sin_kactus} sin match Kactus (ver hoja Conflictos_Dependencia_Area)")
print(f"Discrepancias Cargos ocupados (Kactus) vs Colaboradores activos julio2026: {n_discrepancias} de {n_comparables} comparables")
print(f"Fecha de actualización: VACÍA (tarea no cerrada)")
print(f"Salida final (único archivo del pipeline): {TABLA_FINAL_XLSX}")
print(f"  Hojas: Depuracion_Cargos, QA_Conciliacion, Conflictos_Dependencia_Area")
